In [1]:
import os

# Même méthode que 02_nettoyage_historique : on remonte à la racine du dépôt,
# donc les chemins sont les mêmes que dans src/ (ET-04 : pas de chemin absolu)
while not os.path.isdir("src"):
    os.chdir("..")

import pandas as pd

historique = pd.read_parquet("data/processed/historique_gazole_idf.parquet")
stations = pd.read_parquet("data/processed/stations_idf.parquet")
print("historique :", historique.shape, "| stations :", stations.shape)

# Même type des deux côtés, sinon "75001001" ne retrouve jamais 75001001
print("type de la clé :", historique["id_station"].dtype, "/", stations["id_station"].dtype)

# validate="many_to_one" lèvera une erreur si une station apparaît deux fois dans le flux
print("id en double dans le flux :", stations["id_station"].duplicated().sum())

# Ampleur attendue d'ANO-13, calculée avant la jointure pour la vérifier après
ids_historique = pd.Series(historique["id_station"].unique())
absents_du_flux = ~ids_historique.isin(stations["id_station"])
print("stations dans l'historique :", len(ids_historique))
print("absentes du flux :", absents_du_flux.sum(), f"({absents_du_flux.mean():.1%})")

historique : (587082, 6) | stations : (833, 7)
type de la clé : str / str
id en double dans le flux : 0
stations dans l'historique : 908
absentes du flux : 121 (13.3%)


In [2]:
from src.cleaning import joindre_historique_stations

historique_joint = joindre_historique_stations(historique, stations)
print("relevés :", len(historique), "->", len(historique_joint))

# Contrôle EF-06 : on doit retrouver les 121 stations du bloc 1
sans_ville = historique_joint[historique_joint["ville"].isna()]
print("stations non retrouvées :", sans_ville["id_station"].nunique())
print("relevés concernés :", len(sans_ville), f"({len(sans_ville) / len(historique_joint):.1%})")

relevés : 587082 -> 587082
stations non retrouvées : 121
relevés concernés : 45649 (7.8%)


### Écart de département entre l'historique et le flux (ANO-12)

Avant d'appliquer D-18, on a comparé le département de l'historique (2 premiers chiffres du code postal) et celui du flux v2 (`code_departement`), uniquement pour les stations retrouvées dans le flux.

| id_station | code_postal | Département historique | Département flux |
|---|---|---|---|
| 78800001 | 78800 | 78 | 95 |
| 92120004 | 92120 | 92 | 94 |
| 94160001 | 94160 | 94 | 75 |
| 94542003 | 94542 | 94 | 91 |

**Résultat** : 4 stations, 802 relevés sur 587 082 (≈ 0,14 %), ce sont les 4 stations d'ANO-12.

**Traitement (D-18)** : `joindre_historique_stations()` retient le département du flux quand la station y figure ; pour les 121 stations absentes du flux (ANO-13), le département reste celui du code postal, seule information disponible.

**Impact** : faible sur l'ensemble de la région, mais la station 94160001 est rattachée à Paris (75), ce qui compte pour H-2.